# Credit Risk Classification

This notebook is an end-to-end reproducible presentation for the assignment. The positive class is `bad/default-risk`. The model prioritizes manual review and does not make an automatic lending decision.

## 1. Business framing

The decision is which applications receive enhanced manual review under a maximum capacity of 1,600 reviews per 8,000 monthly applications. The South German Credit target is a historical proxy, not a directly observed 12-month default target.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data_contract.py').exists() and (candidate / 'data' / 'raw').exists():
            return candidate
    raise FileNotFoundError('Could not locate the assignment project root')

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_contract import load_raw_data, make_xy, contract_summary
frame = load_raw_data(ROOT / 'data' / 'raw' / 'SouthGermanCredit.asc')
X, y = make_xy(frame)
contract_summary(frame)


## 1b. Regenerate the analysis artifacts

Run this cell after a clean installation to rebuild Waves 0-5 from the immutable raw source. The frozen test is not used for selection.

In [ ]:
from src.build_waves_0_3 import main as build_waves_0_3
from src.run_wave4 import main as run_wave4
from src.run_wave5 import main as run_wave5
build_waves_0_3()
run_wave4()
run_wave5()


In [ ]:
oof = pd.read_csv(ROOT / 'artifacts' / 'tables' / 'wave-4-oof-threshold-table-all-models.csv')
test = pd.read_csv(ROOT / 'artifacts' / 'tables' / 'wave-4-selected-model-frozen-test-thresholds.csv')
oof[oof['policy'].eq('recommended_capacity_constrained')]


## 2. Classification evaluation

Report accuracy, precision, recall and F1 together. Accuracy is secondary because the review queue is capacity-constrained. The threshold policy is selected on development out-of-fold predictions and confirmed once on the frozen test set.

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1', 'monthly_reviews', 'net_screening_value']
selected_model = json.loads((ROOT / 'artifacts' / 'manifests' / 'wave-4-policy.json').read_text())['selected_model']
oof[oof['model'].eq(selected_model)][['policy', 'threshold'] + [m for m in metrics if m in oof.columns]]


In [ ]:
sensitivity = pd.read_csv(ROOT / 'artifacts' / 'tables' / 'wave-5-sensitivity-development.csv')
sensitivity[['scenario', 'net_screening_value', 'cost_per_expected_default_prevented']]


## 3. Recommendation and limitations

Use the selected model only as a ranking aid for the exact top 20% manual-review queue. Do not auto-approve or auto-reject. The model and policy remain subject to calibration, subgroup review, performance-improvement experiments and independent review.